In [3]:
import json
import sys
from pathlib import Path

# 1. Setup Root di progetto e sys.path
project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import (
    LLM_MODEL,
    EMBEDDING_MODEL,
    DEFAULT_KEEP_ALIVE,
    DEFAULT_NUM_THREAD,
    OLLAMA_URL,
    QDRANT_URL,
)
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_ollama import ChatOllama
from qdrant_client import QdrantClient

COLLECTION_NAME = "ds1_graphrag_chunks"

# 2. Connessione diretta a Qdrant Server tramite URL di configurazione
client = QdrantClient(url=QDRANT_URL)

# Ispezione collezioni sul server
collections = [c.name for c in client.get_collections().collections]
print(f">> Connessione a Qdrant Server: {QDRANT_URL}")
print(f">> Collezioni trovate sul server: {collections}")

if COLLECTION_NAME not in collections:
    client.close()
    raise RuntimeError(
        f"!!>> La collezione '{COLLECTION_NAME}' non esiste sul server Qdrant ({QDRANT_URL}).\n"
        f"<<< Assicurati che i servizi di backend siano attivi ed esegui la cella di ingestione in 01_IngestDataset.ipynb."
    )

# 3. Setup VectorStore e LLM (Ollama)
embeddings = FastEmbedEmbeddings(model_name=EMBEDDING_MODEL)

vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
    validate_collection_config=False,
)

llm = ChatOllama(
    model=LLM_MODEL,
    base_url=OLLAMA_URL,
    keep_alive=DEFAULT_KEEP_ALIVE,
    num_thread=DEFAULT_NUM_THREAD,
    temperature=0
)

print(f"!> Client attivo | LLM: '{LLM_MODEL}' | Embedding: '{EMBEDDING_MODEL}'")

# 4. Caricamento query di test dal benchmark JSON
queries_path = project_root / "data/queries/ds1/eval_queries.json"
with open(queries_path, "r", encoding="utf-8") as f:
    benchmark_queries = json.load(f)

target_ids = ["Q_SINGLE_01", "Q_GLOBAL_01"]
test_queries = [q for q in benchmark_queries if q.get("id") in target_ids]

if not test_queries:
    for cat in ["single_hop", "global_search"]:
        q_found = next((q for q in benchmark_queries if q.get("category") == cat), None)
        if q_found:
            test_queries.append(q_found)

# 5. Esecuzione RAG e analisi comparativa
try:
    for q_obj in test_queries:
        q_id = q_obj.get("id", "N/A")
        category = q_obj.get("category", "N/A").upper()
        query_text = q_obj["query"]

        print("\n" + "=" * 80)
        print(f"<<< TEST QUERY [{q_id}] | CATEGORIA: {category}")
        print(f"<? Domanda: '{query_text}'")
        print("=" * 80)

        # Retrieval dei Top-3 chunk
        results = vector_store.similarity_search_with_score(query=query_text, k=3)

        retrieved_contexts = []
        print("\n-> CHUNK RECUPERATI DA QDRANT:")
        for idx, (doc, score) in enumerate(results, 1):
            metadata = doc.metadata or {}
            source_file = metadata.get("source", metadata.get("file_name", "N/A"))
            snippet = doc.page_content[:180].replace("\n", " ")
            retrieved_contexts.append(doc.page_content)

            print(f"  [{idx}] Score: {score:.4f} | Fonte: {source_file}")
            print(f"      Snippet: {snippet}...\n")

        # Invocazione Ollama
        context_block = "\n\n---\n\n".join(retrieved_contexts)
        prompt = f"""Rispondi alla domanda seguente basandoti ESCLUSIVAMENTE sul contesto fornito.
Se le informazioni nel contesto non sono sufficienti per una risposta completa, dichiaralo apertamente.

CONTESTO:
{context_block}

DOMANDA:
{query_text}

RISPOSTA:"""

        print(f"T> Generazione risposta con Ollama ({LLM_MODEL})...")
        response = llm.invoke(prompt)

        print("\n?> RISPOSTA GENERATA:")
        print(response.content)

        # Confronto sull'accuratezza
        print("\n VALUTAZIONE PRECISIONE:")
        if "GLOBAL" in category or "SUMMARY" in category:
            print("!!!> RISULTATO PARZIALE (Global Query):")
            print("   Il Vector RAG ha estratto solo frammenti locali basati sulla similarità di parole chiave,")
            print("   dimostrando l'incapacità del retrieval vettoriale classico di sintetizzare concetti distribuiti.")
        else:
            print("!!!> RISULTATO PRECISO (Single-Hop Query):")
            print("   Il recupero vettoriale ha individuato correttamente il chunk specifico contenente la risposta.")

finally:
    client.close()
    print("\n<<< Connessione Qdrant Server chiusa correttamente.")

>> Connessione a Qdrant Server: http://localhost:6333
>> Collezioni trovate sul server: ['ds1_graphrag_chunks', 'tesi_graphrag_chunks', 'test_rag_chunks']
!> Client attivo | LLM: 'llama3.2' | Embedding: 'nomic-ai/nomic-embed-text-v1.5'

<<< TEST QUERY [Q_SINGLE_01] | CATEGORIA: SINGLE_HOP
<? Domanda: 'Qual è la funzione dei reflection token nell'architettura Self-RAG?'

-> CHUNK RECUPERATI DA QDRANT:
  [1] Score: 0.6113 | Fonte: /home/jovyan/tesi_graphrag/data/raw/ds1/Self_RAG.pdf
      Snippet: consistent with Menick et al. (2022). Human annotators also find ISREL and ISSUP reflection token predictions are mostly aligned with their assessments. Appendix Table 6 shows seve...

  [2] Score: 0.5749 | Fonte: /home/jovyan/tesi_graphrag/data/raw/ds1/Self_RAG.pdf
      Snippet: to enhance overall generation quality, factuality, and verifiability. consistently retrieves a fixed number of documents for generation regardless of the retrieval necessity (e.g.,...

  [3] Score: 0.5748 | Fonte: /ho